# Governed Agent Learning 0.8

This notebook applies Agent Governance Toolkit controls to learned-policy and Bayesian Agent Learning decisions, then validates and promotes the resulting policy candidate. It uses only in-memory storage and local policy checks.

In [ ]:
import secrets
import sys
import tempfile
from pathlib import Path

# Make this repository checkout importable without installing the package.
cwd = Path.cwd().resolve()
repo = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / "AGENTS.md").exists()),
    cwd,
)
gov_source = repo / "agent-governance-python" / "agent-learning" / "src"
agent_os_source = repo / "agent-governance-python" / "agent-os" / "src"
for source in (gov_source, agent_os_source):
    if source.is_dir():
        sys.path.insert(0, str(source))

# Repository validation may use a source checkout of microsoft/agent-learning v0.8.0.
upstream_source = Path(tempfile.gettempdir()) / "agent-learning-0.8.0-source" / "src"
if upstream_source.is_dir():
    sys.path.insert(0, str(upstream_source))

from agent_learning import (
    Action,
    CaptureConfig,
    DecisionAuthority,
    DecisionCriterion,
    DecisionFrame,
    DecisionOption,
    EvidencePoint,
    InMemoryStore,
    LearningRunner,
    Reward,
    RewardSource,
    SoftmaxPolicy,
    TaskPolicy,
)
from agent_os.lite import govern

from agent_learning_gov import (
    GovernedEpisodeCapture,
    GovernedLearningRunner,
    GovernedPolicyPromotion,
    InMemoryAuditSink,
    LearningGovernanceDashboardModel,
    PromotionStage,
)

In [ ]:
AGENT_ID = "notebook-agent"
TASK_ID = "choose-summary-strategy"
store = InMemoryStore()
audit = InMemoryAuditSink()
provenance_key = secrets.token_bytes(32)
kernel = govern(deny=["delete_records"])
policy = SoftmaxPolicy.from_actions(
    [
        Action(id="grounded_summary", parameters={"role": "user", "estimated_cost": 0.02}),
        Action(id="fast_summary", parameters={"role": "user", "estimated_cost": 0.01}),
    ],
    agent_id=AGENT_ID,
    task_id=TASK_ID,
)
parent = policy.snapshot()
store.store_policy(parent)
capture = GovernedEpisodeCapture(
    kernel,
    config=CaptureConfig(enabled=True, agent_id=AGENT_ID, task_id=TASK_ID),
    store=store,
    audit_sink=audit,
    provenance_key=provenance_key,
)

In [ ]:
for index in range(5):
    action_id = "grounded_summary" if index else "fast_summary"
    task_policy = TaskPolicy(policy.snapshot())
    decision = task_policy.adjudicate(
        task_policy.decide(selected_action_id=action_id),
        "accept",
    )
    context = capture.start(
        f"Summarize account {index}",
        decision_result=decision,
        intent_summary="summarize an account",
        action_type="workflow",
        expected_outcome="return a grounded summary",
    )
    grounded = context.action_id == "grounded_summary"
    episode = capture.end(
        context,
        "Grounded summary" if grounded else "Fast summary",
        execution_status="completed",
        result_summary="returned a summary",
    )
    store.store_reward(
        Reward(
            episode_id=episode.id,
            agent_id=episode.agent_id,
            source=RewardSource.AGGREGATE,
            value=0.85 if grounded else 0.25,
        )
    )

len(store.query_episodes(AGENT_ID, task_id=TASK_ID))

## Bayesian decision route

Full-authority Agent Learning resolves a structured `DecisionFrame` with hard constraints, confidence-weighted Bayesian evidence, robust utility, Pareto elimination, and information gain. Its episode is scored and audited, but it is not a behavior-policy sample and therefore has no `action_logprob`.

In [ ]:
full_snapshot = policy.snapshot()
full_snapshot.metadata["decision_authority"] = DecisionAuthority.FULL.value
actions = {action.id: action for action in full_snapshot.actions}
frame = DecisionFrame(
    task="Choose a strategy for a regulated account",
    criteria=[
        DecisionCriterion(id="grounding", weight=0.7),
        DecisionCriterion(id="latency", weight=0.3),
    ],
    constraints=["data_residency"],
    options=[
        DecisionOption(
            action=actions["grounded_summary"],
            constraint_results={"data_residency": True},
            evidence=[
                EvidencePoint(criterion_id="grounding", source="evaluation", support=0.95),
                EvidencePoint(criterion_id="latency", source="telemetry", support=0.70),
            ],
        ),
        DecisionOption(
            action=actions["fast_summary"],
            constraint_results={"data_residency": True},
            evidence=[
                EvidencePoint(criterion_id="grounding", source="evaluation", support=0.45),
                EvidencePoint(criterion_id="latency", source="telemetry", support=0.90),
            ],
        ),
    ],
)
task_policy = TaskPolicy(full_snapshot)
bayesian_decision = task_policy.decide(frame)
if bayesian_decision.status.value != "resolved":
    bayesian_decision = task_policy.adjudicate(bayesian_decision, "accept")
context = capture.start(
    "Summarize the regulated account",
    decision_result=bayesian_decision,
    intent_summary="summarize a regulated account",
    action_type="workflow",
    expected_outcome="return a compliant summary",
)
bayesian_episode = capture.end(
    context,
    "Grounded regulated-account summary",
    execution_status="completed",
    result_summary="returned a compliant summary",
)
store.store_reward(
    Reward(
        episode_id=bayesian_episode.id,
        agent_id=AGENT_ID,
        source=RewardSource.AGGREGATE,
        value=0.90,
    )
)
assert bayesian_episode.action_logprob is None
bayesian_decision.selection_basis, bayesian_decision.selected_action.id

In [ ]:
learning = LearningRunner(store=store, policy=policy, metrics=[])
governed_runner = GovernedLearningRunner(
    kernel,
    learning,
    audit_sink=audit,
    provenance_key=provenance_key,
)
run = governed_runner.run_offline_batch(
    AGENT_ID,
    task_id=TASK_ID,
    score_missing=False,
)
candidate = governed_runner.last_candidate
report = run.metrics["governance"]
assert run.metrics["episodes_used"] == 5
assert report["bayesian_episodes"] == 1
assert report["reinforce_eligible_episodes"] == 5
assert store.get_active_policy(AGENT_ID, TASK_ID).id == parent.id
report

In [ ]:
deployments = []


def deploy(snapshot, stage, context):
    deployments.append({"policy_id": snapshot.id, "stage": stage.value, **context})
    return deployments[-1]


promoter = GovernedPolicyPromotion(
    kernel,
    store=store,
    audit_sink=audit,
    provenance_key=provenance_key,
    deploy_callback=deploy,
)
canary = promoter.promote_if_compliant(
    candidate,
    stage=PromotionStage.CANARY,
    baseline={"violation_rate": 0.0},
    deployment_context={"traffic_percent": 5},
)
production = promoter.promote_if_compliant(
    candidate.id,
    agent_id=AGENT_ID,
    task_id=TASK_ID,
    stage=PromotionStage.PRODUCTION,
    baseline={"violation_rate": 0.0},
)
assert canary.promoted and production.promoted
assert store.get_active_policy(AGENT_ID, TASK_ID).id == candidate.id
deployments

In [ ]:
dashboard = LearningGovernanceDashboardModel(store, audit_sink=audit).snapshot(
    AGENT_ID,
    task_id=TASK_ID,
)
{
    "summary": dashboard.summary,
    "episode_count": len(dashboard.episodes),
    "policy_count": len(dashboard.policies),
    "audit_event_count": len(dashboard.audit_events),
}

## Result

The learned-policy episodes supplied behavior-policy propensities to REINFORCE. The Bayesian episode remained available for reward scoring, governance evaluation, lineage, and audit, while the runner explicitly excluded it from the policy-gradient update. The candidate stayed inactive through validation and canary rollout; only successful production deployment changed the active-policy pointer.